In [1]:
import sys
import joblib
from pathlib import Path

PROJECT_ROOT = Path().resolve().parents[0]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

import pandas as pd
from sklearn.metrics import accuracy_score, f1_score

from config import PROCESSED_DATA_DIR, MODELS_DIR
from src import model_utils, text_preprocessing

In [2]:
reviews_df = pd.read_parquet(PROCESSED_DATA_DIR / "amazon_reviews_features.parquet")
reviews_df = model_utils.drop_neutral_ratings(reviews_df)
reviews_df["rating"] = reviews_df["rating"].apply(model_utils.classify_rating)
reviews_df["tfidf_text"] = reviews_df["full_text"].apply(text_preprocessing.clean_text)

train_df, val_df, test_df = model_utils.split_data(reviews_df)

model = joblib.load(MODELS_DIR / "binary_logreg_tuned.joblib")

In [3]:
val_features = model_utils.drop_non_feature_columns(val_df.copy())
X_val = val_features.drop(columns=["rating"])
y_val = val_features["rating"]

val_preds = model.predict(X_val)
val_probs = model.predict_proba(X_val)[:, 1]

analysis_df = val_df.copy()
analysis_df["pred_label"] = val_preds
analysis_df["prob_good"] = val_probs
analysis_df["confidence"] = analysis_df["prob_good"].where(
    analysis_df["prob_good"] >= 0.5,
    1 - analysis_df["prob_good"],
)
analysis_df["correct"] = analysis_df["rating"] == analysis_df["pred_label"]
analysis_df["error_type"] = "Correct"
analysis_df.loc[(analysis_df["rating"] == 0) & (analysis_df["pred_label"] == 1), "error_type"] = "False Positive"
analysis_df.loc[(analysis_df["rating"] == 1) & (analysis_df["pred_label"] == 0), "error_type"] = "False Negative"
analysis_df["text_char_length"] = analysis_df["full_text"].str.len()
analysis_df["text_length_bucket"] = pd.qcut(
    analysis_df["text_char_length"],
    q=4,
    duplicates="drop",
).astype(str)
analysis_df["review_count_bucket"] = pd.qcut(
    analysis_df["review_count"],
    q=4,
    duplicates="drop",
).astype(str)

analysis_df[["rating", "pred_label", "prob_good", "confidence", "error_type"]].head()

,rating,pred_label,prob_good,confidence,error_type
7465,0,0,0.025521,0.974479,Correct
9616,0,0,0.000129,0.999871,Correct
4953,0,0,0.007321,0.992679,Correct
6587,0,0,0.027085,0.972915,Correct
15118,0,0,0.109935,0.890065,Correct


## 1. Misclassification Pattern Review

In [4]:
analysis_df["error_type"].value_counts()

error_type
Correct           3831
False Positive     111
False Negative      92
Name: count, dtype: int64

In [5]:
false_positives = analysis_df.loc[analysis_df["error_type"] == "False Positive", [
    "rating",
    "pred_label",
    "prob_good",
    "confidence",
    "review_title",
    "full_text",
]].sort_values("confidence", ascending=False)

false_positives.head(10)

,rating,pred_label,prob_good,confidence,review_title,full_text
1191,0,1,0.998247,0.998247,best company because its a online e.commerce p...,best company because its a online e.commerce p...
16405,0,1,0.996192,0.996192,LOVE Amazon,"LOVE Amazon LOVE Amazon, however their prices ..."
20928,0,1,0.995018,0.995018,Great place for books...but slowly destroying ...,Great place for books...but slowly destroying ...
8994,0,1,0.987303,0.987303,It would be great if I could get my…,It would be great if I could get my… It would ...
16084,0,1,0.986949,0.986949,Not good chop,Not good chop Review text not found
17041,0,1,0.976399,0.976399,Good service,Good service Review text not found
17042,0,1,0.976275,0.976275,Good service,Good service Review text not found
11146,0,1,0.968130,0.968130,The great customer is no more there,The great customer is no more there I have bee...
283,0,1,0.968031,0.968031,Had an issue with a order not really…,Had an issue with a order not really… Had an i...
3037,0,1,0.959780,0.959780,FROM THE RIVER TO THE SEA,FROM THE RIVER TO THE SEA FROM THE RIVER TO TH...


In [6]:
false_negatives = analysis_df.loc[analysis_df["error_type"] == "False Negative", [
    "rating",
    "pred_label",
    "prob_good",
    "confidence",
    "review_title",
    "full_text",
]].sort_values("confidence", ascending=False)

false_negatives.head(10)

,rating,pred_label,prob_good,confidence,review_title,full_text
2615,1,0,0.000184,0.999816,"Just sheer incompetence being displayed there,...","Just sheer incompetence being displayed there,..."
15441,1,0,0.005461,0.994539,"Overall, great!","Overall, great! I haven’t been a prime member ..."
4411,1,0,0.013232,0.986768,Well got out replacement TV from Amazon😱,Well got out replacement TV from Amazon😱 Well ...
16829,1,0,0.013693,0.986307,Everyone in my Household is an Amazon…,Everyone in my Household is an Amazon… Everyo...
20006,1,0,0.016583,0.983417,they give an excellant service,they give an excellant service Whenever I buy ...
16041,1,0,0.017268,0.982732,I would just like to voice a complaint,I would just like to voice a complaint I would...
5942,1,0,0.026337,0.973663,I’m IMPRESSED,I’m IMPRESSED I’m IMPRESSED So someone has bee...
9663,1,0,0.026403,0.973597,You want honesty and REAL facts? Read this. Th...,You want honesty and REAL facts? Read this. Th...
6833,1,0,0.030506,0.969494,I cancelled an item and the refund only…,I cancelled an item and the refund only… I can...
16052,1,0,0.031301,0.968699,ordered a package from USA to eu and…,ordered a package from USA to eu and… ordered ...


## 2. Confidence Analysis

In [7]:
analysis_df.groupby("correct")["confidence"].agg(["count", "mean", "median", "min", "max"])

,count,mean,median,min,max
correct,,,,,
False,203,0.743116,0.751724,0.500746,0.999816
True,3831,0.945272,0.985595,0.502510,0.999997


In [8]:
high_conf_errors = analysis_df.loc[
    (~analysis_df["correct"]) & (analysis_df["confidence"] >= 0.90),
    ["rating", "pred_label", "prob_good", "confidence", "review_title", "full_text"],
].sort_values("confidence", ascending=False)

high_conf_errors.head(10)

,rating,pred_label,prob_good,confidence,review_title,full_text
2615,1,0,0.000184,0.999816,"Just sheer incompetence being displayed there,...","Just sheer incompetence being displayed there,..."
1191,0,1,0.998247,0.998247,best company because its a online e.commerce p...,best company because its a online e.commerce p...
16405,0,1,0.996192,0.996192,LOVE Amazon,"LOVE Amazon LOVE Amazon, however their prices ..."
20928,0,1,0.995018,0.995018,Great place for books...but slowly destroying ...,Great place for books...but slowly destroying ...
15441,1,0,0.005461,0.994539,"Overall, great!","Overall, great! I haven’t been a prime member ..."
8994,0,1,0.987303,0.987303,It would be great if I could get my…,It would be great if I could get my… It would ...
16084,0,1,0.986949,0.986949,Not good chop,Not good chop Review text not found
4411,1,0,0.013232,0.986768,Well got out replacement TV from Amazon😱,Well got out replacement TV from Amazon😱 Well ...
16829,1,0,0.013693,0.986307,Everyone in my Household is an Amazon…,Everyone in my Household is an Amazon… Everyo...
20006,1,0,0.016583,0.983417,they give an excellant service,they give an excellant service Whenever I buy ...


In [9]:
borderline_cases = analysis_df.loc[
    analysis_df["confidence"].between(0.50, 0.60),
    ["rating", "pred_label", "prob_good", "confidence", "review_title", "full_text"],
].sort_values("confidence")

borderline_cases.head(10)

,rating,pred_label,prob_good,confidence,review_title,full_text
3035,0,1,0.500746,0.500746,Amazon is 1 out of 10,Amazon is 1 out of 10 Amazon is so inconvenien...
17731,0,0,0.497490,0.502510,Hopeless delivery service,Hopeless delivery service I ordered 5 items on...
17978,0,0,0.496532,0.503468,AmaGONE!,AmaGONE! Even though Amazon has great items at...
4529,0,1,0.503926,0.503926,I think Amazon supplied me with a fake…,I think Amazon supplied me with a fake… I thin...
8610,1,0,0.493716,0.506284,I have to give a 5 star Glowing Review…,I have to give a 5 star Glowing Review… I have...
9710,0,1,0.506526,0.506526,Absurd Handling & Packaging,Absurd Handling & Packaging Until recent month...
14150,1,1,0.507067,0.507067,I order few household item in the last…,I order few household item in the last… I orde...
14603,0,0,0.492254,0.507746,Amazon arrogance personified!!,Amazon arrogance personified!! Amazon arroganc...
17526,1,0,0.492060,0.507940,Could you please update jwellery…,Could you please update jwellery… Could you pl...
13332,0,1,0.507994,0.507994,I called just to give a compliment to a…,I called just to give a compliment to a… I cal...


## 3. Slice-Based Performance

In [10]:
def slice_metrics(df: pd.DataFrame, slice_col: str) -> pd.DataFrame:
    rows = []
    for slice_value, group in df.groupby(slice_col, dropna=False):
        rows.append({
            slice_col: slice_value,
            "n_reviews": len(group),
            "accuracy": accuracy_score(group["rating"], group["pred_label"]),
            "macro_f1": f1_score(group["rating"], group["pred_label"], average="macro"),
        })
    return pd.DataFrame(rows).sort_values("macro_f1")

In [11]:
slice_metrics(analysis_df, "text_length_bucket")

,text_length_bucket,n_reviews,accuracy,macro_f1
3,"(621.75, 4905.0]",1009,0.973241,0.877514
2,"(345.0, 621.75]",1003,0.961117,0.921169
0,"(15.999, 178.0]",1018,0.928291,0.921219
1,"(178.0, 345.0]",1004,0.936255,0.925198


In [12]:
slice_metrics(analysis_df, "review_count_bucket")

,review_count_bucket,n_reviews,accuracy,macro_f1
2,"(8.0, 326.0]",954,0.949686,0.935933
0,"(0.999, 3.0]",2213,0.950294,0.937443
1,"(3.0, 8.0]",867,0.948097,0.943711


In [13]:
slice_metrics(analysis_df, "country_grouped")

,country_grouped,n_reviews,accuracy,macro_f1
7,IT,25,0.880000,0.880000
0,AU,39,0.948718,0.901515
8,NL,50,0.940000,0.909584
6,IN,114,0.921053,0.911757
3,DK,65,0.938462,0.917092
5,IE,36,0.972222,0.920879
9,OTHER,366,0.939891,0.937182
10,US,1756,0.952733,0.938578
4,GB,1424,0.948736,0.940672
1,CA,132,0.977273,0.961173
